In [21]:
import sys
sys.path.append("../")
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from prompts.routing_prompts import routing_prompt
from models.routing import router, RouteDecision

CONFIDENCE_THRESHOLD = 0.65



In [22]:
# get model for classification
# get prompt for classification
# augment query with prompt
# generate output

In [23]:
def generate(query: str) -> RouteDecision:

    try:
        return router.complete(query).raw

    except Exception as e:
        
        print(f"Routing failed: {e}")

        return RouteDecision(
            route="faq",
            confidence=0.0,
        )

In [24]:
def make_prompt(query) -> str:
    return routing_prompt.render(query=query)

In [25]:
def apply_confidence_threshold(response: RouteDecision, threshold):
    if response.confidence < threshold:
        return None, response.confidence

    return response.route, response.confidence

In [26]:
def process_and_print_query(query: str, correct_label: str, route, confidence) -> dict:
    is_correct = route == correct_label

    result = {
        "query": query,
        "expected": correct_label,
        "predicted": route,
        "confidence": round(confidence, 4) if confidence is not None else None,
        "is_correct": is_correct,
    }

    print(f"Query: {query}")
    print(f"Expected: {correct_label}")
    print(f"Predicted: {route}")

    if confidence is not None:
        print(f"Confidence: {confidence:.2f}")

    print(f"Correct: {'yes' if is_correct else 'no'}")
    print("-" * 60)
    return result



In [27]:
def get_class_query(query) -> str:
    prompt = make_prompt(query)
    response = generate(prompt)
    route, confidence = apply_confidence_threshold(response, CONFIDENCE_THRESHOLD)
    return route, confidence


In [28]:
queries = [
    'How old are you?',
    'Tell me a joke about online shopping.',
    'Translate \"blue shirt\" into Polish.',
    'What is the weather in Warsaw today?',
    'Solve 17 * 23 for me.',
    'Who won the football World Cup in 2018?',
    'Summarize Dune in two sentences.',
    'What is your return policy?', 
    'How can I reset my password?',
    'Can I change the email address connected to my account?',
    'How do I track my parcel after it is shipped?',
    'What should I do if my package arrived damaged?',
    'How long does it usually take to get a refund?',
    'How can I download an invoice for my order?',
    'Give me three examples of blue T-shirts you have available.', 
    'How can I contact the user support?', 
    'Do you have blue Dresses?',
    'Show me black ankle boots in size 39.',
    'Find a beige trench coat for spring.',
    'I need a leather handbag for work meetings.',
    'Show me white sneakers for men.',
    'Recommend accessories for a navy evening gown.',
    'Create a look suitable for a wedding party happening during dawn.'
]



In [29]:
labels = [
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    'faq',
    'faq',
    'faq',
    'faq',
    'faq',
    'faq',
    'faq',
    'product',
    'faq',
    'product',
    'product',
    'product',
    'product',
    'product',
    'product',
    'product'
]



In [30]:
def _label_name(label) -> str:
    return "none" if label is None else str(label)


def _safe_divide(numerator: float, denominator: float) -> float:
    return numerator / denominator if denominator else 0.0


def _to_markdown_table(df: pd.DataFrame) -> str:
    columns = list(df.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    rows = []

    for record in df.to_dict(orient="records"):
        rendered = []
        for column in columns:
            value = record.get(column)
            if value is None or pd.isna(value):
                rendered.append("-")
            elif isinstance(value, float):
                rendered.append(f"{value:.3f}")
            else:
                rendered.append(str(value))
        rows.append("| " + " | ".join(rendered) + " |")

    return "\n".join([header, separator, *rows])


def build_markdown_report(results_df: pd.DataFrame, output_path: str = "../reports/class_routing_evaluation.md"):
    classes = [None, "faq", "product"]
    confusion_rows = []
    class_metrics_rows = []

    for expected in classes:
        row = {"expected": _label_name(expected)}
        for predicted in classes:
            row[_label_name(predicted)] = int(((results_df["expected"] == expected) & (results_df["predicted"] == predicted)).sum())
        confusion_rows.append(row)

    confusion_df = pd.DataFrame(confusion_rows)

    for label in classes:
        tp = int(((results_df["expected"] == label) & (results_df["predicted"] == label)).sum())
        fp = int(((results_df["expected"] != label) & (results_df["predicted"] == label)).sum())
        fn = int(((results_df["expected"] == label) & (results_df["predicted"] != label)).sum())
        support = int((results_df["expected"] == label).sum())

        precision = _safe_divide(tp, tp + fp)
        recall = _safe_divide(tp, tp + fn)
        f1 = _safe_divide(2 * precision * recall, precision + recall)

        class_metrics_rows.append(
            {
                "label": _label_name(label),
                "precision": round(precision, 3),
                "recall": round(recall, 3),
                "f1": round(f1, 3),
                "support": support,
            }
        )

    class_metrics_df = pd.DataFrame(class_metrics_rows)

    summary_df = pd.DataFrame(
        [
            {
                "total_cases": int(len(results_df)),
                "accuracy": round(float(results_df["is_correct"].mean()), 3),
                "avg_confidence": round(float(results_df["confidence"].fillna(0).mean()), 3),
            }
        ]
    )

    per_case_df = results_df.copy()
    per_case_df["expected"] = per_case_df["expected"].map(_label_name)
    per_case_df["predicted"] = per_case_df["predicted"].map(_label_name)

    lines = [
        "# Class Routing Report",
        "",
        "Source notebook: `notebooks/06_class_routing.ipynb`",
        "",
        "## Summary",
        "",
        f"- Total cases: {int(summary_df.iloc[0]['total_cases'])}",
        f"- Accuracy: {summary_df.iloc[0]['accuracy']:.3f}",
        f"- Average confidence: {summary_df.iloc[0]['avg_confidence']:.3f}",
        "",
        "## Final Metrics",
        "",
        _to_markdown_table(summary_df),
        "",
        "## Per-class Metrics",
        "",
        _to_markdown_table(class_metrics_df),
        "",
        "## Per-case Results",
        "",
    ]

    for index, row in per_case_df.iterrows():
        lines.extend(
            [
                f"### Case {index + 1}",
                "",
                f"- Query: {row['query']}",
                f"- Expected: {row['expected']}",
                f"- Predicted: {row['predicted']}",
                f"- Confidence: {row['confidence']:.3f}" if pd.notna(row['confidence']) else "- Confidence: -",
                f"- Correct: {'yes' if row['is_correct'] else 'no'}",
                "",
            ]
        )

    lines.extend([
        "## Full Results Table",
        "",
        _to_markdown_table(per_case_df),
        "",
    ])

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text("\n".join(lines), encoding="utf-8")
    return output, summary_df, class_metrics_df, confusion_df, per_case_df


results = []

for query, correct_label in zip(queries, labels):
    route, confidence = get_class_query(query)
    results.append(process_and_print_query(query, correct_label, route, confidence))

results_df = pd.DataFrame(results)
report_path, summary_df, class_metrics_df, confusion_df, per_case_df = build_markdown_report(results_df)

display(Markdown(f"Markdown report saved to: `{report_path}`"))



Query: How old are you?
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Tell me a joke about online shopping.
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Translate "blue shirt" into Polish.
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: What is the weather in Warsaw today?
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Solve 17 * 23 for me.
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Who won the football World Cup in 2018?
Expected: None
Predicted: None
Confidence: 0.10
Correct: yes
------------------------------------------------------------
Query: Summarize Dune in two sentenc

Markdown report saved to: `../reports/class_routing_evaluation.md`